# Markets vs Polls: Who Called It Better? (2024 US presidential, state level)
Logic: `sql/forecasts.sql`, `src/forecast_scoring.py`.

In [1]:
import sys; sys.path.insert(0,'../src')
from forecast_scoring import *
con=connect(); d=matched_panel(con); d.groupby('days_before').size()

days_before
1     35
7     34
14    26
21    23
28     7
dtype: int64

## Brier and log loss by horizon (matched states only)

In [2]:
score(d).round(4)

,days_before,source,states,brier,log_loss,correct
0,1,polymarket,35,0.0415,0.1505,33
1,1,kalshi,35,0.0428,0.1688,33
2,1,polls,35,0.0528,0.1607,31
3,7,polymarket,34,0.0290,0.1268,34
4,7,kalshi,34,0.0339,0.1419,33
5,7,polls,34,0.0473,0.1497,31
6,14,polymarket,26,0.0380,0.1622,26
7,14,kalshi,26,0.0427,0.1732,26
8,14,polls,26,0.0612,0.1907,23
9,21,polymarket,23,0.0592,0.2153,22


## Paired bootstrap, 1 day out (negative = first source better)

In [3]:
g=d[d.days_before==1]
{k:paired_bootstrap(g[a],g[b],g.rep_won) for k,(a,b) in {'poly-polls':('polymarket','polls'),'kalshi-polls':('kalshi','polls'),'poly-kalshi':('polymarket','kalshi')}.items()}

{'poly-polls': (-0.011337661371641636,
  (np.float64(-0.022602366724926257), np.float64(-0.001879835479243329))),
 'kalshi-polls': (-0.010031647085927341,
  (np.float64(-0.022466229604101797), np.float64(0.0009207730644576956))),
 'poly-kalshi': (-0.0013060142857142958,
  (np.float64(-0.0040426326785714395), np.float64(0.0016233721428571407)))}

## Calibration (T-1, pooled sources)

In [4]:
{s: calibration_table(g[s], g.rep_won) for s in ['polymarket','kalshi','polls']}

{'polymarket':                 n      pred  actual
 bin                                
 (-0.001, 0.2]  12  0.050167     0.0
 (0.2, 0.4]      1  0.385000     1.0
 (0.4, 0.6]      3  0.523333     1.0
 (0.6, 0.8]      3  0.703333     1.0
 (0.8, 1.0]     16  0.961438     1.0,
 'kalshi':                 n      pred  actual
 bin                                
 (-0.001, 0.2]  12  0.082500     0.0
 (0.2, 0.4]      1  0.370000     1.0
 (0.4, 0.6]      3  0.530000     1.0
 (0.6, 0.8]      4  0.740000     1.0
 (0.8, 1.0]     15  0.953333     1.0,
 'polls':                 n      pred  actual
 bin                                
 (-0.001, 0.2]  12  0.025639     0.0
 (0.2, 0.4]      1  0.369867     1.0
 (0.4, 0.6]      5  0.499189     1.0
 (0.6, 0.8]      1  0.637572     1.0
 (0.8, 1.0]     16  0.979621     1.0}